In [0]:
from pyspark.sql import functions as F, Window

spark.sql("USE plworkforce_catalog.003_gold")

# Load pre-aggregated data cubes

cube_financial = spark.table("plworkforce_catalog.003_gold.cube_financial")
cube_hr_workforce = spark.table("plworkforce_catalog.003_gold.cube_hr_workforce")

print("Data cubes loaded successfully")

In [0]:
# KPI 1 - Monthly Consolidated Revenue
# Filter only revenue rows from the financial cube
monthly_revenue = (
    cube_financial
    .filter(F.col("account_type") == "Revenue")     # Only revenue
    .groupBy("year_month")                          # Consolidated across BOTH companies
    .agg(F.sum("revenue").alias("total_revenue"))   # Total monthly revenue
    .orderBy("year_month")
)

# Window for Month-over-Month comparison
w = Window.orderBy("year_month")

# Add previous month + MoM growth %
result_revenue = (
    monthly_revenue
    .withColumn("prev_month", F.lag("total_revenue").over(w))
    .withColumn(
        "mom_growth_pct",
        F.round(((F.col("total_revenue") - F.col("prev_month")) / F.col("prev_month")) * 100, 2)
    )
    .withColumn("total_revenue", F.round("total_revenue", 2))
    .withColumn("prev_month", F.round("prev_month", 2))
)

display(result_revenue)


In [0]:
# KPI 2: Cost of Sales by Month

cost_of_sales = (
    cube_financial
    .filter(F.col("account_type") == "Cogs")     # Only direct cost accounts
    .groupBy("company_id", "company_name", "year_month")
    .agg(
        F.round(F.sum("cogs"), 2).alias("cost_of_sales")   # Total monthly COGS
    )
    .orderBy("company_id", "year_month")        # Sort by company_id
)

display(cost_of_sales)

In [0]:

# KPI 3: Monthly Gross Profit Margin

# Revenue by company,month
rev = (
    cube_financial
    .filter(F.col("account_type") == "Revenue")
    .groupBy("company_id", "company_name", "year_month")
    .agg(F.sum("revenue").alias("revenue"))
)

# COGS by company,month
cogs = (
    cube_financial
    .filter(F.col("account_type") == "Cogs")
    .groupBy("company_id", "year_month")
    .agg(F.sum("cogs").alias("cogs"))
)

#  calculate GP,GPM
gpm = (
    rev.join(cogs, ["company_id", "year_month"], "left")
    .withColumn("gross_profit", F.col("revenue") - F.col("cogs"))
    .withColumn(
        "gross_profit_margin_pct",
        F.round(F.try_divide(F.col("gross_profit"), F.col("revenue")) * 100, 2)
    )
    .select(
        "company_name",
        "year_month",
        F.round("revenue", 2).alias("revenue"),
        F.round("cogs", 2).alias("cogs"),
        F.round("gross_profit", 2).alias("gross_profit"),
        "gross_profit_margin_pct"
    )
    .orderBy("company_name", "year_month")
)

display(gpm)

In [0]:
# KPI 4: Operating Expense Breakdown

opex_breakdown = (
    cube_financial
    .filter(F.col("account_type") == "Expense")      
    .groupBy(
        "company_id",
        "company_name",
        "category"                                   
    )
    .agg(
        F.round(F.sum("operating_expenses"), 2).alias("total_operating_expense")
    )
    .orderBy("company_name", "category")
)

display(opex_breakdown)

In [0]:
# KPI 5: Average Compensation by Position 
avg_comp = (
    cube_hr_workforce
    .groupBy("company_name", "position")
    .agg(
        F.round(F.avg("total_compensation_sum"), 2).alias("avg_total_compensation"),
        F.round(F.avg("total_base_salary"), 2).alias("avg_base_salary"),
        F.round(F.avg("total_bonus"), 2).alias("avg_bonus"),
        F.round(F.avg("total_overtime"), 2).alias("avg_overtime"),
        F.round(F.avg("total_commission"), 2).alias("avg_commission")
    )
    .orderBy("company_name", "position")
)

display(avg_comp)

In [0]:

# KPI 6: Net Profit by Month

# Revenue per company per month
rev = (
    cube_financial
    .filter(F.col("account_type") == "Revenue")
    .groupBy("company_id", "company_name", "year_month")
    .agg(F.sum("revenue").alias("revenue"))
)

# COGS per company per month
cogs = (
    cube_financial
    .filter(F.col("account_type") == "Cogs")
    .groupBy("company_id", "year_month")
    .agg(F.sum("cogs").alias("cogs"))
)

#  Operating Expenses per company per month
opex = (
    cube_financial
    .filter(F.col("account_type") == "Expense")
    .groupBy("company_id", "year_month")
    .agg(F.sum("operating_expenses").alias("opex"))
)

# Net Profit Calculation
net_profit = (
    rev.join(cogs, ["company_id", "year_month"], "left")
       .join(opex, ["company_id", "year_month"], "left")
       .withColumn("gross_profit", F.col("revenue") - F.col("cogs"))
       .withColumn("net_profit", F.col("gross_profit") - F.col("opex"))
       .withColumn(
            "net_profit_margin_pct",
            F.round(F.try_divide(F.col("net_profit"), F.col("revenue")) * 100, 2)
       )
       .select(
            "company_name",
            "year_month",
            F.round("revenue", 2).alias("revenue"),
            F.round("cogs", 2).alias("cogs"),
            F.round("opex", 2).alias("opex"),
            F.round("net_profit", 2).alias("net_profit"),
            "net_profit_margin_pct"
       )
       .orderBy("company_name", "year_month")
)

display(net_profit)

In [0]:
# KPI 7: Overtime and Bonus Analysis 

overtime_bonus_analysis = (
    cube_hr_workforce
    .groupBy("company_name", "department_name")
    .agg(
        F.sum("total_overtime").alias("total_overtime"),
        F.sum("total_bonus").alias("total_bonus"),
        F.sum("total_base_salary").alias("total_base_salary"),
        F.sum("headcount").alias("employee_count")
    )
    .withColumn("overtime_to_base_ratio_pct",
                F.round(F.try_divide(F.col("total_overtime"),
                                     F.col("total_base_salary")) * 100, 2))
    .withColumn("avg_overtime_per_employee",
                F.round(F.try_divide(F.col("total_overtime"),
                                     F.col("employee_count")), 2))
    .withColumn("avg_bonus_per_employee",
                F.round(F.try_divide(F.col("total_bonus"),
                                     F.col("employee_count")), 2))
    .orderBy("company_name", F.desc("total_overtime"))
)

display(overtime_bonus_analysis)

In [0]:
# KPI 8: Cost per Department 

dept_cost = (
    cube_hr_workforce
    .groupBy("company_name", "department_name")
    .agg(
        F.sum("total_compensation_sum").alias("department_cost"),
        F.sum("headcount").alias("employee_count")
    )
    .withColumn(
        "cost_per_employee",
        F.round(F.try_divide(F.col("department_cost"), F.col("employee_count")), 2)
    )
    .select(
        "company_name",
        "department_name",
        F.round("department_cost", 2).alias("department_cost"),
        "employee_count",
        "cost_per_employee"
    )
    .orderBy("company_name", F.desc("department_cost"))
)

display(dept_cost)

In [0]:
# KPI 9: Headcount Distribution by Department

headcount_distribution = (
    cube_hr_workforce
    .filter(F.col("employee_status") == "ACTIVE")     
    .groupBy("company_name", "department_name")      
    .agg(
        F.sum("headcount").alias("active_employee_count")
    )
    .orderBy("company_name", F.desc("active_employee_count"))
)

display(headcount_distribution)

In [0]:

# KPI 10: Payroll Cost as % of Company Revenue

# Payroll cost per company per month
payroll_monthly = (
    cube_hr_workforce
    .groupBy("company_id", "company_name", "year_month")
    .agg(F.sum("total_compensation_sum").alias("payroll_cost"))
)

# Revenue per company per month
revenue_monthly = (
    cube_financial
    .filter(F.col("account_type") == "Revenue")
    .groupBy("company_id", "year_month")
    .agg(F.sum("revenue").alias("total_revenue"))
)

# Payroll % of Revenue
payroll_revenue_ratio = (
    payroll_monthly.join(revenue_monthly, ["company_id", "year_month"], "left")
    .withColumn(
        "payroll_to_revenue_pct",
        F.round(F.try_divide(F.col("payroll_cost"), F.col("total_revenue")) * 100, 2)
    )
    .select(
        "company_name",
        "year_month",
        F.round("total_revenue", 2).alias("total_revenue"),
        F.round("payroll_cost", 2).alias("payroll_cost"),
        "payroll_to_revenue_pct"
    )
    .orderBy("company_name", "year_month")
)

display(payroll_revenue_ratio)